# Exercise 2: PyTorch core

In this exercise you’ll build core PyTorch “muscle memory” that you’ll reuse in basically every model you write:

- **Autograd**: how gradients are created, how they accumulate, and how to compute gradients for one or multiple inputs.
- **Dataloading**: writing small `Dataset`s, using `DataLoader`, and custom `collate_fn`.
- **Optimizers**: implementing **AdamW** updates from scratch (state, bias correction, weight decay).
- **Training basics**: a clean single training step.
- **Initialization**: fan-in/out and common initializers (Xavier / Kaiming), plus a helper to init `nn.Linear`.

As before: fill in all `TODO`s without changing function names or signatures.
When debugging, print shapes/dtypes/devices, and write tiny sanity checks (e.g. compare to PyTorch’s built-ins).


In [ ]:
from __future__ import annotations
from dataclasses import dataclass
import torch
from torch import nn

## Autograd fundamentals

PyTorch builds a computation graph when you apply operations to tensors with `requires_grad=True`.
Calling `backward()` (or `torch.autograd.grad`) computes gradients by traversing that graph.

### Key concepts
- **Leaf tensor**: a tensor created by you (not the result of an operation) with `requires_grad=True`. Leaf tensors can store gradients in `.grad`.
- **Gradient accumulation**: calling `backward()` adds into `.grad` (it does not overwrite). You must reset gradients between steps/calls.
- **`torch.autograd.grad` vs `.backward()`**
  - `torch.autograd.grad(f, x)` returns `df/dx` directly and does not write into `x.grad` unless you explicitly do so.
  - `f.backward()` writes gradients into `.grad` of leaf tensors.

In the next functions you’ll compute gradients for a simple scalar function such as `f(x) = sum(x^2)` using both APIs.

### `torch.no_grad()`
Wrap inference-only code to avoid tracking gradients and building graphs:
- saves memory
- speeds up evaluation

### `detach()`
`y = x.detach()` returns a tensor that shares data with `x` but is **not connected** to the autograd graph.
This is useful when you want to treat something as a constant target.

### `model.train()` vs `model.eval()`
- `train()` enables training behavior (e.g. dropout active, batchnorm updates running stats).
- `eval()` enables inference behavior (e.g. dropout off, batchnorm uses running stats).

In [ ]:
def grad_with_autograd_grad(x: torch.Tensor) -> torch.Tensor:
    """
    Compute gradient of f(x) = sum(x^2) using torch.autograd.grad

    Requirements:
    - Do not call .backward().
    - x should require grad inside the function (don't assume it does).
    - Must return df/dx
    """
    x = x.detach().clone().requires_grad_(True)
    f = (x ** 2).sum()
    return torch.autograd.grad(f, x)[0]

x_g = torch.tensor([1.0, 2.0, 3.0])
print(grad_with_autograd_grad(x_g))  # tensor([2., 4., 6.])  == 2*x

In [ ]:
def test_grad_with_autograd_grad():
    x = torch.tensor([1.0, 2.0, 3.0])
    g = grad_with_autograd_grad(x)
    assert torch.allclose(g, 2 * x), f"expected 2x, got {g}"
    assert x.grad is None, "must not mutate caller's x"

    y = torch.tensor([4.0, -1.0], requires_grad=True)
    g2 = grad_with_autograd_grad(y)
    assert torch.allclose(g2, 2 * y.detach()), "must work even if x already requires grad"
    print("grad_with_autograd_grad: OK")

test_grad_with_autograd_grad()

In [ ]:

def grad_with_backward(x: torch.Tensor) -> torch.Tensor:
    """
    Compute gradient of f(x) = sum(x^2) using .backward().

    Requirements:
    - Must return df/dx
    - Must not leak gradients across calls (watch x.grad accumulation)
    """
    x = x.detach().clone().requires_grad_(True)
    f = (x ** 2).sum()
    f.backward()
    return x.grad.clone()

x_b = torch.tensor([1.0, 2.0, 3.0])
print(grad_with_backward(x_b))  # tensor([2., 4., 6.])
print(grad_with_backward(x_b))  # same result again, not tensor([4., 8., 12.])

In [ ]:
def test_grad_with_backward():
    x = torch.tensor([1.0, 2.0, 3.0])
    g1 = grad_with_backward(x)
    g2 = grad_with_backward(x)
    assert torch.allclose(g1, 2 * x) and torch.allclose(g2, 2 * x), "must not accumulate across calls"
    assert x.grad is None, "must not mutate caller's x"
    print("grad_with_backward: OK")

test_grad_with_backward()

In [ ]:
def grad_wrt_multiple_inputs(
    a: torch.Tensor, b: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Compute gradients w.r.t. multiple inputs. The function is f(a, b) = sum(a^2 + ab).

    Return:
        (df/da, df/db)

    Requirements:
    - Use torch.autograd.grad
    - Ensure both a and b require grad in this function.
    """
    # TODO: implement
    raise NotImplementedError

<details>
<summary>💡 Solution</summary>

```python
def grad_wrt_multiple_inputs(a, b):
    a = a.detach().clone().requires_grad_(True)
    b = b.detach().clone().requires_grad_(True)
    f = (a ** 2 + a * b).sum()
    da, db = torch.autograd.grad(f, (a, b))
    return da, db
```

`torch.autograd.grad(f, (a, b))` takes a tuple of inputs and returns a tuple of gradients in the same order. Cloning + `requires_grad_(True)` keeps the caller's tensors untouched, same trick as `grad_with_autograd_grad`.
</details>

In [ ]:
def test_grad_wrt_multiple_inputs():
    a = torch.tensor([1.0, 2.0])
    b = torch.tensor([3.0, 4.0])
    da, db = grad_wrt_multiple_inputs(a, b)
    assert torch.allclose(da, 2 * a + b), da
    assert torch.allclose(db, a), db
    assert a.grad is None and b.grad is None, "must not mutate caller's a/b"
    print("grad_wrt_multiple_inputs: OK")

test_grad_wrt_multiple_inputs()

## Dataloading

In PyTorch, a `Dataset` defines how to fetch a *single* training example, and a `DataLoader` handles:
- batching
- shuffling
- parallel workers
- optional custom batching logic via `collate_fn`

### `Dataset` in one sentence
A `Dataset` only needs:
- `__len__`: number of items
- `__getitem__`: return one item (e.g. `(x, y)`)

### Why `collate_fn` matters
The default DataLoader collation stacks items along a new batch dimension.
That works for fixed-size tensors, but it breaks for **variable-length sequences**.

So we’ll implement padding ourselves:
- Convert a list of 1D token sequences into a padded tensor `(B, T_max)`
- Track `lengths` and a `padding_mask`

### Mask convention for padding
For padding masks in this exercise:
- `padding_mask[b, t] == True` means **this is padding / invalid**
- `padding_mask[b, t] == False` means **this is a real token**

In [7]:
from torch.utils.data import DataLoader, Dataset

In [10]:
class TensorPairDataset(Dataset):
    """
    Minimal dataset wrapping (x, y).

    x: (N, ...)
    y: (N, ...)

    N is the number of samples. The dataset should return tuples of (x[i], y[i]).
    """

    def __init__(self, x: torch.Tensor, y: torch.Tensor):
        assert x.shape[0] == y.shape[0]
        self.x = x
        self.y = y

    def __len__(self) -> int:
        return self.x.shape[0]

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        return (self.x[idx], self.y[idx])

<details>
<summary>💡 Solution</summary>

```python
class TensorPairDataset(Dataset):
    def __init__(self, x, y):
        assert x.shape[0] == y.shape[0]
        self.x = x
        self.y = y

    def __len__(self):
        return self.x.shape[0]

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]
```

A `Dataset` only needs `__len__` and `__getitem__`; `__init__` just stores the tensors.
</details>

In [11]:
def test_tensor_pair_dataset():
    x = torch.arange(6).reshape(3, 2)
    y = torch.arange(3)
    ds = TensorPairDataset(x, y)
    assert len(ds) == 3
    xi, yi = ds[1]
    assert torch.equal(xi, x[1]) and torch.equal(yi, y[1])
    print("TensorPairDataset: OK")

test_tensor_pair_dataset()

TensorPairDataset: OK


In [16]:
class NextTokenDataset(Dataset):
    """
    Next-token prediction dataset.

    Given tokens of shape (N, T), produce:
      input_ids  = tokens[:, :-1]
      target_ids = tokens[:, 1:]

    Return per item:
      (input_ids, target_ids)

    Notes:
    - Returned tensors should be 1D of length (T-1).
    - dtype should remain integer.
    """

    def __init__(self, tokens: torch.Tensor):
        self.tokens = tokens

    def __len__(self) -> int:
        return self.tokens.shape[0]

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        seq = self.tokens[idx]
        return seq[:-1], seq[1:]

<details>
<summary>💡 Solution</summary>

```python
class NextTokenDataset(Dataset):
    def __init__(self, tokens):
        self.tokens = tokens

    def __len__(self):
        return self.tokens.shape[0]

    def __getitem__(self, idx):
        seq = self.tokens[idx]
        return seq[:-1], seq[1:]
```

Slicing `seq[:-1]` / `seq[1:]` on a 1D row gives the shifted input/target pair directly.
</details>

In [17]:
def test_next_token_dataset():
    tokens = torch.arange(12).reshape(3, 4)
    ds = NextTokenDataset(tokens)
    assert len(ds) == 3
    inp, tgt = ds[0]
    assert inp.dim() == 1 and inp.shape[0] == 3, "must be 1D of length T-1"
    assert torch.equal(inp, torch.tensor([0, 1, 2]))
    assert torch.equal(tgt, torch.tensor([1, 2, 3]))
    print("NextTokenDataset: OK")

test_next_token_dataset()

NextTokenDataset: OK


In [18]:

class RandomCropSequenceDataset(Dataset):
    """
    Sequence dataset that returns random crops of fixed length.

    tokens: (N, T_total)
    crop_len: L

    For each __getitem__:
      - sample a start index s so that s+L <= T_total
      - return tokens[idx, s:s+L]

    Requirements:
    - Use a torch.Generator for deterministic behavior if seed is provided.
    - Do NOT use Python's random module.
    """

    def __init__(self, tokens: torch.Tensor, crop_len: int, seed: int | None = None):
        assert crop_len <= tokens.shape[1]
        self.tokens = tokens
        self.crop_len = crop_len
        self.g = torch.Generator()
        if seed is not None:
            self.g.manual_seed(seed)

    def __len__(self) -> int:
        return self.tokens.shape[0]

    def __getitem__(self, idx: int) -> torch.Tensor:
        T_total = self.tokens.shape[1]
        max_start = T_total - self.crop_len
        s = torch.randint(0, max_start + 1, (1,), generator=self.g).item()
        return self.tokens[idx, s:s+self.crop_len]

<details>
<summary>💡 Solution</summary>

```python
class RandomCropSequenceDataset(Dataset):
    def __init__(self, tokens, crop_len, seed=None):
        assert crop_len <= tokens.shape[1]
        self.tokens = tokens
        self.crop_len = crop_len
        self.generator = torch.Generator()
        if seed is not None:
            self.generator.manual_seed(seed)

    def __len__(self):
        return self.tokens.shape[0]

    def __getitem__(self, idx):
        T_total = self.tokens.shape[1]
        max_start = T_total - self.crop_len
        s = torch.randint(0, max_start + 1, (1,), generator=self.generator).item()
        return self.tokens[idx, s : s + self.crop_len]
```

Use `torch.randint(..., generator=self.generator)` (not Python's `random`) so a seeded generator gives reproducible crops. `max_start = T_total - crop_len` is the last valid start index, so the high bound is `max_start + 1` (`randint`'s high is exclusive).
</details>

In [19]:
def test_random_crop_dataset():
    tokens = torch.arange(20).reshape(2, 10)
    ds1 = RandomCropSequenceDataset(tokens, crop_len=4, seed=0)
    ds2 = RandomCropSequenceDataset(tokens, crop_len=4, seed=0)
    c1, c2 = ds1[0], ds2[0]
    assert c1.shape[0] == 4
    assert torch.equal(c1, c2), "same seed must give the same crop"
    print("RandomCropSequenceDataset: OK")

test_random_crop_dataset()

RandomCropSequenceDataset: OK


In [26]:


@dataclass(frozen=True)
class PaddedBatch:
    """
    A padded batch for variable-length sequences.

    tokens: LongTensor (B, T_max)
    lengths: LongTensor (B,)
    padding_mask: BoolTensor (B, T_max) where True means "this is padding"
    """

    tokens: torch.Tensor
    lengths: torch.Tensor
    padding_mask: torch.Tensor


def pad_1d_sequences(seqs: list[torch.Tensor], pad_value: int = 0) -> PaddedBatch:
    """
    Pad a list of 1D integer tensors to the same length.

    Requirements:
    - Return PaddedBatch(tokens, lengths, padding_mask)
    - padding_mask[b, t] == True iff t >= lengths[b]
    - tokens should be dtype long, if not cast them
    """
    lengths = torch.tensor([s.shape[0] for s in seqs], dtype=torch.long)
    T_max = int(lengths.max().item()) if len(seqs) > 0 else 0
    tokens = torch.full((len(seqs),T_max), pad_value, dtype=torch.long)
    for i, s in enumerate(seqs):
        tokens[i, :s.shape[0]] = s.to(torch.long)
    padding_mask = torch.arange(T_max).unsqueeze(0) >= lengths.unsqueeze(1)
    return PaddedBatch(tokens=tokens, lengths=lengths, padding_mask=padding_mask)
            

<details>
<summary>💡 Solution</summary>

```python
def pad_1d_sequences(seqs, pad_value=0):
    lengths = torch.tensor([s.shape[0] for s in seqs], dtype=torch.long)
    t_max = int(lengths.max().item()) if len(seqs) > 0 else 0
    tokens = torch.full((len(seqs), t_max), pad_value, dtype=torch.long)
    for i, s in enumerate(seqs):
        tokens[i, : s.shape[0]] = s.to(torch.long)
    padding_mask = torch.arange(t_max).unsqueeze(0) >= lengths.unsqueeze(1)
    return PaddedBatch(tokens=tokens, lengths=lengths, padding_mask=padding_mask)
```

`torch.arange(t_max).unsqueeze(0) >= lengths.unsqueeze(1)` broadcasts `(1, T_max)` against `(B, 1)` to give the `(B, T_max)` mask in one shot — position `t` is padding once `t >= lengths[b]`.
</details>

In [27]:
def test_pad_1d_sequences():
    seqs = [torch.tensor([1, 2, 3]), torch.tensor([4, 5])]
    batch = pad_1d_sequences(seqs, pad_value=0)
    assert batch.tokens.dtype == torch.long
    assert torch.equal(batch.tokens, torch.tensor([[1, 2, 3], [4, 5, 0]]))
    assert torch.equal(batch.lengths, torch.tensor([3, 2]))
    assert torch.equal(
        batch.padding_mask,
        torch.tensor([[False, False, False], [False, False, True]]),
    )
    print("pad_1d_sequences: OK")

test_pad_1d_sequences()

pad_1d_sequences: OK


In [28]:
def collate_next_token_batch(
    batch: list[tuple[torch.Tensor, torch.Tensor]], pad_value: int = 0
) -> dict[str, torch.Tensor]:
    """
    Collate for NextTokenDataset samples that may have variable lengths.

    batch: list of (input_ids, target_ids), each 1D

    Return dict with:
      - input_ids: (B, T_max)
      - target_ids: (B, T_max)
      - attention_mask: (B, T_max) where True means "keep" (NOT padding)
      - padding_mask: (B, T_max) where True means "padding"

    Requirements:
    - pad input_ids and target_ids consistently
    - attention_mask is the logical NOT of padding_mask
    """
    input_batch = pad_1d_sequences([ids for ids, _ in batch], pad_value)
    target_batch = pad_1d_sequences([tgt for _, tgt in batch], pad_value)
    padding_mask = input_batch.padding_mask
    return {
        "input_ids": input_batch.tokens,
        "target_ids": target_batch.tokens,
        "attention_mask": ~padding_mask,
        "padding_mask": padding_mask
    }
        

<details>
<summary>💡 Solution</summary>

```python
def collate_next_token_batch(batch, pad_value=0):
    input_batch = pad_1d_sequences([ids for ids, _ in batch], pad_value)
    target_batch = pad_1d_sequences([tgt for _, tgt in batch], pad_value)
    padding_mask = input_batch.padding_mask
    return {
        "input_ids": input_batch.tokens,
        "target_ids": target_batch.tokens,
        "attention_mask": ~padding_mask,
        "padding_mask": padding_mask,
    }
```

Reuse `pad_1d_sequences` for both halves of the pair. `input_ids`/`target_ids` come from the same original sequence shifted by one, so they're always the same length per sample — one `padding_mask` covers both.
</details>

In [29]:
def test_collate_next_token_batch():
    tokens = torch.tensor([[1, 2, 3, 4], [5, 6, 7, 8]])
    ds = NextTokenDataset(tokens)
    out = collate_next_token_batch([ds[0], ds[1]], pad_value=0)
    assert torch.equal(out["input_ids"], torch.tensor([[1, 2, 3], [5, 6, 7]]))
    assert torch.equal(out["target_ids"], torch.tensor([[2, 3, 4], [6, 7, 8]]))
    assert torch.equal(out["attention_mask"], ~out["padding_mask"])
    print("collate_next_token_batch: OK")

test_collate_next_token_batch()

collate_next_token_batch: OK


In [30]:
def make_dataloader(
    dataset: Dataset,
    batch_size: int,
    shuffle: bool = True,
    drop_last: bool = False,
    collate_fn=None,
    num_workers: int = 0,
) -> DataLoader:
    """
    Create a DataLoader with optional collate_fn.
    """
    return DataLoader(dataset, batch_size, shuffle, drop_last=drop_last, collate_fn=collate_fn, num_workers=num_workers)

<details>
<summary>💡 Solution</summary>

```python
def make_dataloader(dataset, batch_size, shuffle=True, drop_last=False, collate_fn=None, num_workers=0):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        collate_fn=collate_fn,
        num_workers=num_workers,
    )
```

Thin passthrough — `DataLoader` already does batching/shuffling/collation; this just forwards the arguments.
</details>

In [31]:
def test_make_dataloader():
    x = torch.arange(10).reshape(5, 2)
    y = torch.arange(5)
    ds = TensorPairDataset(x, y)
    dl = make_dataloader(ds, batch_size=2, shuffle=False)
    batches = list(dl)
    assert len(batches) == 3, "5 items, batch_size=2 -> 3 batches (last one smaller)"
    xb, yb = batches[0]
    assert xb.shape == (2, 2)
    print("make_dataloader: OK")

test_make_dataloader()

make_dataloader: OK


## Optimizers (AdamW from scratch)

PyTorch optimizers keep **state** for each parameter (e.g. moment estimates in Adam).
In this section you’ll implement **AdamW**, which is Adam + *decoupled* weight decay.

### AdamW state
For each parameter tensor `p` we store:
- `m`: first moment (EMA of gradients)
- `v`: second moment (EMA of squared gradients)
- `t`: step counter

### Update overview (high level)
1) Update moments `m, v`
2) Bias-correct them (`m_hat, v_hat`)
3) Apply parameter update:
   `p -= lr * ( m_hat / (sqrt(v_hat) + eps) + weight_decay * p )`

Notes:
- This update is **in-place** (mutates `p`).
- Gradients should not be modified.
- State tensors must match parameter shape/device/dtype.

In [34]:
@dataclass
class AdamWState:
    """
    Per-parameter AdamW state.

    m: first moment
    v: second moment
    t: step count
    """

    m: torch.Tensor
    v: torch.Tensor
    t: int


def init_adamw_state(p: torch.Tensor) -> AdamWState:
    """
    Initialize AdamW state tensors for a parameter tensor p.

    What to create:
    - m: zeros like p, same shape/device/dtype
    - v: zeros like p, same shape/device/dtype
    - t: step counter starting at 0

    Notes / requirements:
    - Use torch.zeros_like(p) for m and v.
    - Do NOT attach gradients to the state (initialize under torch.no_grad()).
    - t starts at 0. In adamw_step_, increment t to 1 on the first update *before*
      computing bias correction terms (1 - beta1^t) and (1 - beta2^t).
    - State tensors must live on the same device as p (CPU vs GPU) and have the
      same dtype as p.
    """
    with torch.no_grad():
      m = torch.zeros_like(p)
      v = torch.zeros_like(p)
    t = 0
    return AdamWState(m=m, v=v, t=t)

      

<details>
<summary>💡 Solution</summary>

```python
def init_adamw_state(p):
    with torch.no_grad():
        m = torch.zeros_like(p)
        v = torch.zeros_like(p)
    return AdamWState(m=m, v=v, t=0)
```

`torch.zeros_like(p)` copies shape/dtype/device automatically. The `no_grad()` block is just defensive — `zeros_like` doesn't track grad by default anyway, but it documents intent.
</details>

In [35]:
def test_init_adamw_state():
    p = torch.randn(3, 4)
    state = init_adamw_state(p)
    assert torch.equal(state.m, torch.zeros_like(p)) and torch.equal(state.v, torch.zeros_like(p))
    assert state.m.dtype == p.dtype and state.m.device == p.device
    assert state.t == 0
    assert not state.m.requires_grad and not state.v.requires_grad
    print("init_adamw_state: OK")

test_init_adamw_state()

init_adamw_state: OK


In [40]:
def adamw_step_(
    p: torch.Tensor,
    grad: torch.Tensor,
    state: AdamWState,
    lr: float,
    betas: tuple[float, float] = (0.9, 0.999),
    eps: float = 1e-8,
    weight_decay: float = 0.01,
) -> AdamWState:
    """
    In-place AdamW parameter update (updates p).

    Algorithm (AdamW):
      m = beta1*m + (1-beta1)*grad
      v = beta2*v + (1-beta2)*grad^2
      m_hat = m / (1 - beta1^t)
      v_hat = v / (1 - beta2^t)
      p = p - lr * (m_hat / (sqrt(v_hat) + eps) + weight_decay * p)

    Requirements:
    - Update p in-place.
    - Return updated state (with incremented t).
    - Do not modify grad.
    - Should work for any tensor shape.
    """
    beta1, beta2 = betas
    state.t += 1
    with torch.no_grad():
        state.m.mul_(beta1).add_(grad, alpha=1-beta1)
        state.v.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
        m_hat = state.m / (1 - beta1 ** state.t)
        v_hat = state.v / (1 - beta2 ** state.t)
        p.sub_(lr * (m_hat / (v_hat.sqrt() + eps) + weight_decay * p))
    return state

<details>
<summary>💡 Solution</summary>

```python
def adamw_step_(p, grad, state, lr, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01):
    beta1, beta2 = betas
    state.t += 1
    with torch.no_grad():
        state.m.mul_(beta1).add_(grad, alpha=1 - beta1)
        state.v.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
        m_hat = state.m / (1 - beta1 ** state.t)
        v_hat = state.v / (1 - beta2 ** state.t)
        p.sub_(lr * (m_hat / (v_hat.sqrt() + eps) + weight_decay * p))
    return state
```

Verified against `torch.optim.AdamW` directly: same `p` after N steps on the same grads, to `1e-6`. Key points:
- increment `state.t` **before** computing bias correction, as the docstring says.
- `grad` is only read (`mul_`/`addcmul_` write into `state.m`/`state.v`, not `grad`), so it's never modified.
- everything under `torch.no_grad()` since these are raw in-place tensor ops, not an autograd-tracked computation.
</details>

In [41]:
def test_adamw_step_matches_builtin():
    """Compare our manual AdamW update against torch.optim.AdamW over several steps."""
    torch.manual_seed(0)
    p_manual = torch.randn(4)
    p_torch = torch.nn.Parameter(p_manual.clone())
    opt = torch.optim.AdamW([p_torch], lr=0.1)
    state = init_adamw_state(p_manual)
    grad_before = None
    for _ in range(5):
        grad = torch.randn(4)
        grad_before = grad.clone()
        adamw_step_(p_manual, grad, state, lr=0.1)
        assert torch.equal(grad, grad_before), "must not modify grad"
        p_torch.grad = grad.clone()
        opt.step()
        opt.zero_grad()
    assert torch.allclose(p_manual, p_torch.detach(), atol=1e-6), (p_manual, p_torch.detach())
    assert state.t == 5
    print("adamw_step_ matches torch.optim.AdamW: OK")

test_adamw_step_matches_builtin()

adamw_step_ matches torch.optim.AdamW: OK


In [42]:
def adamw_step_many_(
    params: list[torch.Tensor],
    grads: list[torch.Tensor],
    states: list[AdamWState],
    lr: float,
    betas: tuple[float, float] = (0.9, 0.999),
    eps: float = 1e-8,
    weight_decay: float = 0.01,
) -> list[AdamWState]:
    """
    Apply AdamW to many parameters.

    Requirements:
    - len(params) == len(grads) == len(states)
    - Update each param in-place.
    - Return the list of updated states.
    """
    assert len(params) == len(grads) == len(states)
    for (param, grad, state) in zip(params, grads, states):
        adamw_step_(param, grad, state, lr, betas, eps, weight_decay)
    return states

<details>
<summary>💡 Solution</summary>

```python
def adamw_step_many_(params, grads, states, lr, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01):
    assert len(params) == len(grads) == len(states)
    for p, grad, state in zip(params, grads, states):
        adamw_step_(p, grad, state, lr, betas, eps, weight_decay)
    return states
```

Just loops `adamw_step_` over the three parallel lists via `zip`. `adamw_step_` mutates `p` and `state` in place, so no need to collect anything beyond returning `states`.
</details>

In [43]:
def test_adamw_step_many():
    p1, p2 = torch.zeros(3), torch.zeros(2)
    s1, s2 = init_adamw_state(p1), init_adamw_state(p2)
    g1, g2 = torch.ones(3), torch.ones(2)
    out_states = adamw_step_many_([p1, p2], [g1, g2], [s1, s2], lr=0.1)
    assert out_states[0].t == 1 and out_states[1].t == 1
    assert not torch.equal(p1, torch.zeros(3)), "p1 must have been updated in-place"
    assert not torch.equal(p2, torch.zeros(2)), "p2 must have been updated in-place"
    print("adamw_step_many_: OK")

test_adamw_step_many()

adamw_step_many_: OK


## Training basics

A minimal training step follows the same pattern almost everywhere:

1) set model to train mode
2) reset gradients
3) forward pass
4) compute loss
5) backward pass
6) step optimizer

In this exercise you’ll implement a single MSE training step using a standard PyTorch optimizer.
Return a Python float loss value.

In [ ]:
def train_step_mse(
    model: nn.Module,
    batch: tuple[torch.Tensor, torch.Tensor],
    optimizer: torch.optim.Optimizer,
) -> float:
    """
    One MSE train step using standard torch optimizer.
    """
    # TODO: implement
    raise NotImplementedError

<details>
<summary>💡 Solution</summary>

```python
def train_step_mse(model, batch, optimizer):
    model.train()
    optimizer.zero_grad()
    x, y = batch
    pred = model(x)
    loss = nn.functional.mse_loss(pred, y)
    loss.backward()
    optimizer.step()
    return loss.item()
```

The six steps from the "Training basics" markdown cell, in order: `train()` → `zero_grad()` → forward → loss → `backward()` → `step()`. `.item()` converts the scalar loss tensor to a plain Python float.
</details>

In [ ]:
def test_train_step_mse():
    torch.manual_seed(0)
    model = nn.Linear(3, 1)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
    x = torch.randn(8, 3)
    y = torch.randn(8, 1)
    loss0 = train_step_mse(model, (x, y), optimizer)
    loss1 = train_step_mse(model, (x, y), optimizer)
    assert isinstance(loss0, float)
    assert loss1 < loss0, "loss should decrease after a gradient step on the same batch"
    print("train_step_mse: OK")

test_train_step_mse()

## Parameter initialization

Initialization matters because it controls signal and gradient scales at the start of training.

### Fan-in / fan-out
- `fan_in`: number of input connections to a unit
- `fan_out`: number of output connections from a unit

For a Linear layer weight of shape `(out_features, in_features)`:
- `fan_in = in_features`
- `fan_out = out_features`

### Common schemes
- **Xavier / Glorot** (often good for tanh / linear-ish nets):
  keeps variance stable across layers when activations are roughly symmetric.
- **Kaiming / He** (often good for ReLU-like nets):
  accounts for the fact that ReLU zeroes out about half the inputs.

In this section you’ll implement Xavier uniform and Kaiming uniform and use them to initialize `nn.Linear`.
We also always zero the bias unless explicitly told otherwise.

In [ ]:
def fan_in_fan_out(weight: torch.Tensor) -> tuple[int, int]:
    """Compute (fan_in, fan_out) for a weight tensor."""
    # TODO: implement
    raise NotImplementedError

<details>
<summary>💡 Solution</summary>

```python
def fan_in_fan_out(weight):
    fan_out, fan_in = weight.shape[0], weight.shape[1]
    return fan_in, fan_out
```

Straight from the markdown cell above: for a weight of shape `(out_features, in_features)`, `fan_in = in_features` (dim 1), `fan_out = out_features` (dim 0).
</details>

In [ ]:
def test_fan_in_fan_out():
    w = torch.randn(5, 3)  # nn.Linear(3, 5).weight shape: (out_features, in_features)
    fan_in, fan_out = fan_in_fan_out(w)
    assert (fan_in, fan_out) == (3, 5)
    print("fan_in_fan_out: OK")

test_fan_in_fan_out()

In [ ]:
def xavier_uniform_(weight: torch.Tensor, gain: float = 1.0) -> torch.Tensor:
    """
    In-place Xavier/Glorot uniform init:
      bound = gain * sqrt(6 / (fan_in + fan_out))
      U(-bound, bound)
    """
    # TODO: implement
    raise NotImplementedError

<details>
<summary>💡 Solution</summary>

```python
import math

def xavier_uniform_(weight, gain=1.0):
    fan_in, fan_out = fan_in_fan_out(weight)
    bound = gain * math.sqrt(6 / (fan_in + fan_out))
    with torch.no_grad():
        weight.uniform_(-bound, bound)
    return weight
```

Direct translation of the formula in the docstring. `weight.uniform_(-bound, bound)` is the in-place uniform fill; wrap it in `torch.no_grad()` since `weight` is a parameter that may require grad and this is a raw data mutation, not a tracked op.

(Needs `import math` — add it to the imports cell at the top if not already there.)
</details>

In [ ]:
def test_xavier_uniform_():
    w = torch.empty(100, 50)
    xavier_uniform_(w)
    bound = math.sqrt(6 / (50 + 100))
    assert w.min().item() >= -bound - 1e-6 and w.max().item() <= bound + 1e-6
    assert not w.requires_grad
    print("xavier_uniform_: OK")

test_xavier_uniform_()

In [ ]:
def kaiming_uniform_(weight: torch.Tensor, nonlinearity: str = "relu") -> torch.Tensor:
    """
    In-place Kaiming/He uniform init.

    Follow this common choice:
      gain = sqrt(2) for ReLU
      std = gain / sqrt(fan_in)
      bound = sqrt(3) * std
      U(-bound, bound)
    """
    # TODO: implement
    raise NotImplementedError

<details>
<summary>💡 Solution</summary>

```python
def kaiming_uniform_(weight, nonlinearity="relu"):
    fan_in, _ = fan_in_fan_out(weight)
    gain = math.sqrt(2) if nonlinearity == "relu" else 1.0
    std = gain / math.sqrt(fan_in)
    bound = math.sqrt(3) * std
    with torch.no_grad():
        weight.uniform_(-bound, bound)
    return weight
```

Same shape as `xavier_uniform_`, different formula — only `fan_in` matters here, and the gain depends on the nonlinearity (`sqrt(2)` for ReLU per the docstring).
</details>

In [ ]:
def test_kaiming_uniform_():
    w = torch.empty(100, 50)
    kaiming_uniform_(w, nonlinearity="relu")
    std = math.sqrt(2) / math.sqrt(50)
    bound = math.sqrt(3) * std
    assert w.min().item() >= -bound - 1e-6 and w.max().item() <= bound + 1e-6
    print("kaiming_uniform_: OK")

test_kaiming_uniform_()

In [ ]:
def init_linear_(layer: nn.Linear, scheme: str = "xavier") -> nn.Linear:
    """
    Initialize an nn.Linear in-place.

    scheme:
      - "xavier"
      - "kaiming_relu"
      - "zero" (weights and bias = 0)
    """
    # TODO: implement
    raise NotImplementedError

<details>
<summary>💡 Solution</summary>

```python
def init_linear_(layer, scheme="xavier"):
    with torch.no_grad():
        if scheme == "xavier":
            xavier_uniform_(layer.weight)
        elif scheme == "kaiming_relu":
            kaiming_uniform_(layer.weight, nonlinearity="relu")
        elif scheme == "zero":
            layer.weight.zero_()
        else:
            raise ValueError(f"unknown scheme: {scheme}")
        if layer.bias is not None:
            layer.bias.zero_()
    return layer
```

Dispatches to the two functions above by `scheme`, then always zeros the bias (per the section note: "we also always zero the bias unless explicitly told otherwise").
</details>

In [ ]:
def test_init_linear_():
    layer = nn.Linear(10, 5)
    init_linear_(layer, scheme="zero")
    assert torch.equal(layer.weight, torch.zeros_like(layer.weight))
    assert torch.equal(layer.bias, torch.zeros_like(layer.bias))

    layer2 = nn.Linear(10, 5)
    init_linear_(layer2, scheme="xavier")
    assert torch.equal(layer2.bias, torch.zeros_like(layer2.bias)), "bias always zeroed"
    assert not torch.equal(layer2.weight, torch.zeros_like(layer2.weight))

    layer3 = nn.Linear(10, 5)
    init_linear_(layer3, scheme="kaiming_relu")
    assert not torch.equal(layer3.weight, torch.zeros_like(layer3.weight))
    print("init_linear_: OK")

test_init_linear_()